In [1]:
import pandas as pd
import sqlalchemy
import joblib 
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path


In [2]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [3]:
query = """
SELECT *
FROM customer_features
"""

customer_df = pd.read_sql(
    query,
    engine
)

customer_df.head()

,customer_unique_id,total_orders,total_revenue,avg_order_value,avg_review_score,total_freight,avg_installments,first_purchase,last_purchase,customer_lifetime_days,...,revenue_per_day,high_value_customer,freight_percentage,review_category,recency_group,R_score,F_score,M_score,RFM_score,customer_value_score
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.0,12.00,8.0,2018-05-10 10:56:00,2018-05-10 10:56:00,0,...,141.90,0,8.456660,Excellent,Warm,4,1,4,414,9
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.0,8.29,1.0,2018-05-07 11:11:00,2018-05-07 11:11:00,0,...,27.19,0,30.489150,Average,Warm,4,1,1,411,6
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.0,17.22,8.0,2017-03-10 21:05:00,2017-03-10 21:05:00,0,...,86.22,0,19.972164,Average,Inactive,1,1,2,112,4
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,4.0,17.63,4.0,2017-10-12 20:29:00,2017-10-12 20:29:00,0,...,43.62,0,40.417240,Average,Inactive,2,1,1,211,4
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,5.0,16.89,6.0,2017-11-14 19:45:00,2017-11-14 19:45:00,0,...,196.89,0,8.578394,Excellent,Cold,2,1,4,214,7


In [4]:
segmentation_features = [
    "total_orders",
    "total_revenue",
    "avg_order_value",
    "avg_review_score",
    "customer_lifetime_days",
    "recency_days",
    "customer_tenure_months",
    "revenue_per_day",
    "customer_value_score"
]

X = customer_df[segmentation_features]

In [5]:
X = X.fillna(
    X.median(numeric_only=True)
)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [7]:
from sklearn.cluster import KMeans

inertia = []

for k in range(2, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)

    inertia.append(
        model.inertia_
    )

print(inertia)

[677731.8586232844, 571484.2564095268, 484243.54974671756, 423559.57217133185, 356273.1173576479, 301201.3854366745, 266299.02483502496, 244329.91528927453, 224106.9351991609]


In [8]:
pd.DataFrame({
    "clusters": range(2,11),
    "inertia": inertia
})

,clusters,inertia
0,2,677731.858623
1,3,571484.256410
2,4,484243.549747
3,5,423559.572171
4,6,356273.117358
5,7,301201.385437
6,8,266299.024835
7,9,244329.915289
8,10,224106.935199


In [9]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

customer_df["customer_segment"] = (
    kmeans.fit_predict(
        X_scaled
    )
)

In [10]:
customer_df[
    "customer_segment"
].value_counts()

customer_segment
1    41599
3    35741
0    16254
2     1492
4     1010
Name: count, dtype: int64

In [11]:
segment_summary = (
    customer_df
    .groupby(
        "customer_segment"
    )
    [
        [
            "total_orders",
            "total_revenue",
            "avg_order_value",
            "customer_value_score",
            "recency_days"
        ]
    ]
    .mean()
)

segment_summary

,total_orders,total_revenue,avg_order_value,customer_value_score,recency_days
customer_segment,,,,,
0,1.021226,210.157116,154.096128,9.124523,290.036545
1,1.033342,197.003042,160.418905,10.562465,176.437222
2,1.029491,2713.105918,1356.142017,10.929625,297.109920
3,1.008170,124.699907,109.573096,6.938894,418.155927
4,2.264356,459.195356,145.419700,13.094059,205.780198


In [12]:
segment_map = {
    0: "Regular Customer",
    1: "Active Customer",
    2: "VIP Customer",
    3: "At Risk Customer",
    4: "Loyal Customer"
}

customer_df["segment_name"] = (
    customer_df["customer_segment"]
    .map(segment_map)
)

In [13]:
customer_df[
    [
        "customer_unique_id",
        "customer_segment",
        "segment_name"
    ]
].to_sql(
    "customer_segments",
    con=engine,
    if_exists="replace",
    index=False
)

96

In [14]:
from pathlib import Path
import joblib
BASE_DIR = Path.cwd()

MODEL_DIR = BASE_DIR / "Trained_Models" / "customer_segmentation"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    kmeans,
    MODEL_DIR / "kmeans_model.pkl"
)

['C:\\Users\\vansh\\OneDrive\\Desktop\\sureTrust\\Trained_Models\\customer_segmentation\\kmeans_model.pkl']

In [15]:
from pathlib import Path

BASE_DIR = Path.cwd()

MODEL_DIR = BASE_DIR / "Trained_Models" / "customer_segmentation"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    scaler,
    MODEL_DIR / "segmentation_scaler.pkl"
)

['C:\\Users\\vansh\\OneDrive\\Desktop\\sureTrust\\Trained_Models\\customer_segmentation\\segmentation_scaler.pkl']